# Model Folding Stress Test - ResNet18/CIFAR10

POSTECH Final Project, Spring 2026  
Track 2: Stress test of "Forget the Data and Fine-tuning!" (Wang et al., ICLR 2025)

This notebook reproduces three REPAIR variants (Fold-Naive, Fold-AR, Fold-R) on ResNet18/CIFAR10. It also includes an attempted VGG11-BN extension which did not finish (see the report for details).

**Environment**: Google Colab T4 GPU, Python 3.12, PyTorch 2.x.

**Runtime estimate**:
- ResNet18 training (60 epochs): about 40 min
- 3 REPAIR sweeps: about 1.5 hours total
- VGG11 attempt: did not complete

**AI assistance**: Claude (ChatGPT) was used for debugging the four compatibility patches and for English polishing of the report. All experimental design, training, sweeps, and analysis were done by me.

## Step 1. Check GPU

In [ ]:
!nvidia-smi

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 2. Mount Google Drive

I save the trained checkpoint and results to Drive so they survive Colab session resets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/model_folding_project'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Save dir: {SAVE_DIR}")

## Step 3. Clone the upstream repo

I use the `marza96/ModelFolding` repo at commit `33e2818`, which the universal repo's README recommends for CV reproduction.

In [ ]:
%cd /content
!rm -rf ModelFolding
!git clone https://github.com/marza96/ModelFolding
%cd ModelFolding
!git checkout 33e2818e017c250aacbff62003e83bd87236ed17
!ls

## Step 4. Install missing dependency

The upstream `requirements.txt` is missing `thop`, which the main script imports. I do not install `hartigan-kmeans` because it does not build on Python 3.12 (see Step 5).

In [ ]:
!pip install thop -q
print("thop installed")

## Step 5. Apply four compatibility patches

The released code targets Python 3.8 and older PyTorch, so it does not run on the current Colab as-is. I patched four things:

1. **hartigan-kmeans build failure**: its `versioneer.py` calls `configparser.SafeConfigParser`, removed in Python 3.12. I replace `HKMeans` with `sklearn.cluster.KMeans`. The paper only describes the method as "k-means" so this should not change results much.

2. **`n_jobs=-1` not supported by sklearn**: I drop this argument and set `verbose=0` to reduce log spam.

3. **`torch.nn.modules.module.ModuleAttributeError` removed in PyTorch 2.x**: the except clause in `utils/utils.py` (meant to fall through for ResNet18 BasicBlocks without `bn3`) crashes. I catch the base `AttributeError` instead.

4. **(in Step 7)**: the recommended pretrained checkpoint is not compatible with the marza96 ResNet18 architecture, so I train from scratch.

In [ ]:
# Patch 1 & 2: weight_clustering.py
fp = '/content/ModelFolding/utils/weight_clustering.py'
with open(fp, 'r') as f:
    content = f.read()

# Patch 1
content = content.replace(
    'from hkmeans import HKMeans',
    'from sklearn.cluster import KMeans as HKMeans  # PATCH'
)

# Patch 2
content = content.replace(
    'HKMeans(n_clusters=self.n_clusters, random_state=None, n_init=10,\n                        n_jobs=-1, max_iter=20, verbose=True)',
    'HKMeans(n_clusters=self.n_clusters, random_state=None, n_init=10,\n                        max_iter=20, verbose=0)  # PATCH'
)
with open(fp, 'w') as f:
    f.write(content)

# Patch 3: utils.py
fp = '/content/ModelFolding/utils/utils.py'
with open(fp, 'r') as f:
    content = f.read()
content = content.replace(
    'torch.nn.modules.module.ModuleAttributeError',
    'AttributeError  # PATCH'
)
with open(fp, 'w') as f:
    f.write(content)

print("Patches applied")

## Step 6. Sanity check the model

Before training, check that the patched code imports cleanly and the model has the expected size.

In [ ]:
import sys
sys.path.insert(0, '/content/ModelFolding')

from model.resnet import ResNet18

model = ResNet18()
n_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded. Parameter count: {n_params:,}")
print(f"Expected: 11,173,962")

## Step 7. Why I cannot use the recommended pretrained checkpoint

The universal repo's README recommends the `PyTorch_CIFAR10` ResNet18 checkpoint (reported accuracy 93.07%). All 122 state-dict keys actually match under the `{downsample:shortcut, fc:linear}` mapping the authors provide, but the two models have different forward graphs:

| | PyTorch_CIFAR10 | marza96 |
|---|---|---|
| Initial maxpool | yes | no |
| ReLUs per BasicBlock | 1 | 2 |

Loading the public checkpoint into the marza96 ResNet18 gives only about 21% accuracy before any compression. So instead I train the marza96 ResNet18 from scratch on CIFAR10.

In [ ]:
# Train ResNet18 from scratch on CIFAR10
# Runtime on T4: about 40 minutes
import torch.nn as nn
import torch.optim as optim
from utils.datasets import get_cifar10
from tqdm import tqdm

NUM_EPOCHS = 60
LR = 0.1
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4
BATCH_SIZE = 128

model = ResNet18().cuda()
train_loader = get_cifar10(train=True, bs=BATCH_SIZE)
test_loader = get_cifar10(train=False)

opt = optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=NUM_EPOCHS)
crit = nn.CrossEntropyLoss()

ckpt_path = f'{SAVE_DIR}/resnet18_cifar10_marza96.pt'
print(f"Training for {NUM_EPOCHS} epochs. Best model saved to {ckpt_path}\n")

best_acc = 0
for epoch in range(NUM_EPOCHS):
    model.train()
    for x, y in tqdm(train_loader, desc=f"Ep {epoch+1}/{NUM_EPOCHS}", leave=False):
        x, y = x.cuda(), y.cuda()
        opt.zero_grad()
        loss = crit(model(x), y)
        loss.backward()
        opt.step()
    sched.step()

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.cuda(), y.cuda()
            pred = model(x).argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    acc = 100 * correct / total
    print(f"Epoch {epoch+1:2d}/{NUM_EPOCHS}: test acc = {acc:.2f}%")

    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), ckpt_path)

print(f"\nTraining done. Best acc: {best_acc:.2f}%")

## Step 8. Run three REPAIR variants

I sweep over 14 layer-wise sparsity values for each of the three variants, using the release script as-is (the sweep is hard-coded inside).

- `Fold-AR` (DF_REPAIR): data-free repair (the paper's main idea)
- `Fold-Naive` (NO_REPAIR): no repair (negative control)
- `Fold-R` (REPAIR): data-driven repair (upper bound, uses training data)

Total runtime on T4: about 1.5 hours.

In [ ]:
import os
os.environ["WANDB_MODE"] = "offline"
os.environ["WANDB_SILENT"] = "true"

%cd /content/ModelFolding

CKPT = f"{SAVE_DIR}/resnet18_cifar10_marza96.pt"

# Variant 1: Fold-AR
print("="*60); print("Variant 1/3: Fold-AR (DF_REPAIR)"); print("="*60)
!python resnet18_cifar10_weight_clustering.py \
    --checkpoint {CKPT} \
    --repair "DF_REPAIR" \
    --proj_name "stress_test" --exp_name "baseline_fold_ar"

# Variant 2: Fold-Naive
print("\n" + "="*60); print("Variant 2/3: Fold-Naive (NO_REPAIR)"); print("="*60)
!python resnet18_cifar10_weight_clustering.py \
    --checkpoint {CKPT} \
    --repair "NO_REPAIR" \
    --proj_name "stress_test" --exp_name "baseline_fold_naive"

# Variant 3: Fold-R
print("\n" + "="*60); print("Variant 3/3: Fold-R (REPAIR)"); print("="*60)
!python resnet18_cifar10_weight_clustering.py \
    --checkpoint {CKPT} \
    --repair "REPAIR" \
    --proj_name "stress_test" --exp_name "baseline_fold_r"

print("\n\nAll three sweeps done.")

## Step 9. Collect results and plot

Each run prints `model after adapt: acc:XX.XX%` for each sparsity point. I **manually** copy these numbers into the lists below.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Results from the run logs above
results = {
    'sparsity_target': [
        0.01, 0.025, 0.05, 0.075, 0.10, 0.15, 0.25, 0.35,
        0.45, 0.55, 0.65, 0.75, 0.85, 0.95
    ],
    # Fold-AR (DF_REPAIR)
    'fold_ar': [
        94.58, 94.58, 94.47, 94.32, 93.54, 92.99, 90.97, 81.86,
        70.65, 44.29, 24.25, 12.14,  9.63, 10.39
    ],
    # Fold-Naive (NO_REPAIR)
    'fold_naive': [
        94.49, 94.44, 94.24, 93.67, 93.07, 92.44, 84.42, 69.16,
        28.47, 18.41, 12.83, 10.76, 10.17, 10.00
    ],
    # Fold-R (REPAIR with data)
    'fold_r': [
        94.54, 94.64, 94.42, 94.38, 93.99, 93.74, 92.16, 89.31,
        84.34, 71.44, 56.10, 28.84, 14.79, 10.09
    ],
}

df = pd.DataFrame(results)
df['original_acc'] = 94.57
df.to_csv(f'{SAVE_DIR}/baseline_results.csv', index=False)
print(df.to_string(index=False))

In [ ]:
# Figure 1: main comparison
plt.figure(figsize=(7, 4.5))
plt.plot(df['sparsity_target'], df['fold_naive'], 'o-',
         label='Fold-Naive (no repair)', color='#d62728', linewidth=2, markersize=6)
plt.plot(df['sparsity_target'], df['fold_ar'], 's-',
         label='Fold-AR (data-free)', color='#1f77b4', linewidth=2, markersize=6)
plt.plot(df['sparsity_target'], df['fold_r'], '^-',
         label='Fold-R (with data)', color='#2ca02c', linewidth=2, markersize=6)
plt.axhline(y=10, color='gray', linestyle=':', alpha=0.5, label='Random chance')
plt.axhline(y=94.57, color='black', linestyle='--', alpha=0.4, label='Original (94.57%)')
plt.xlabel('Layer-wise sparsity', fontsize=12)
plt.ylabel('Test accuracy (%)', fontsize=12)
plt.title('Model Folding on ResNet18/CIFAR10: Three REPAIR strategies', fontsize=12)
plt.legend(loc='lower left', fontsize=10)
plt.grid(True, alpha=0.3)
plt.ylim(0, 100)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/fig1_baseline_comparison.png', dpi=150, bbox_inches='tight')
plt.savefig(f'{SAVE_DIR}/fig1_baseline_comparison.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Figure 2: Fold-R minus Fold-AR gap
fig, ax = plt.subplots(figsize=(7, 4.5))
ar = np.array(df['fold_ar'])
r = np.array(df['fold_r'])
gap = r - ar
sp = np.array(df['sparsity_target'])
ax.plot(sp, gap, 'o-', color='#9467bd', linewidth=2, markersize=7)
ax.fill_between(sp, 0, gap, alpha=0.2, color='#9467bd')
ax.set_xlabel('Layer-wise sparsity', fontsize=12)
ax.set_ylabel('Accuracy gap: Fold-R - Fold-AR (%)', fontsize=12)
ax.set_title('Where does the data-free assumption break?', fontsize=11)
ax.axvline(x=0.55, color='red', linestyle='--', alpha=0.5, label='Break point ~ 55%')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/fig2_ar_breakdown.png', dpi=150, bbox_inches='tight')
plt.savefig(f'{SAVE_DIR}/fig2_ar_breakdown.pdf', bbox_inches='tight')
plt.show()

print(f"\nFigures saved to {SAVE_DIR}")

## Step 10. Check saved files

In [ ]:
print(f"Contents of {SAVE_DIR}:\n")
for f in sorted(os.listdir(SAVE_DIR)):
    path = os.path.join(SAVE_DIR, f)
    if os.path.isfile(path):
        size_kb = os.path.getsize(path) / 1024
        if size_kb > 1024:
            print(f"  {f}: {size_kb/1024:.1f} MB")
        else:
            print(f"  {f}: {size_kb:.1f} KB")
    else:
        print(f"  {f}/ (directory)")

## Step 11. Attempted VGG11-BN extension (did not finish)

The proposal commits to running the same experiment on VGG11-BN. The goal was to check whether the ~55% Fold-AR breakdown seen on ResNet18 also happens on a non-residual architecture.

**This attempt did not finish.** Training itself worked, but the VGG sweep did not produce any result even after more than 24 hours on a T4 GPU, and at some point the process appeared to halt without an error. I leave the cells below as-is for documentation: anyone trying to reproduce can see exactly what I attempted.

**Likely reasons**:
- sklearn's KMeans is single-threaded and scales worse on VGG's wider (4096-dim) layers than on ResNet18's at most 512-dim layers.
- The VGG path may simply be less well tested than the ResNet path that the README recommends.

See Section 5 of the report for the full discussion.

In [ ]:
# VGG11-BN training (this part DID work, ~15 min on T4)
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models.vgg import make_layers, VGG
from utils.datasets import get_cifar10
from tqdm import tqdm

vgg11_cfg = [64, 'M', 128, 'M', 256, 256, 'M', 512, 512, 'M', 512, 512, 'M']
features = make_layers(vgg11_cfg, batch_norm=True)
model = VGG(features=features, num_classes=10).cuda()

NUM_EPOCHS = 25
LR = 0.05
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4
BATCH_SIZE = 128

train_loader = get_cifar10(train=True, bs=BATCH_SIZE)
test_loader = get_cifar10(train=False)

opt = optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=NUM_EPOCHS)
crit = nn.CrossEntropyLoss()

vgg_ckpt_path = f"{SAVE_DIR}/vgg11_bn_cifar10_torchvision.pt"
best_acc = 0.0
for epoch in range(NUM_EPOCHS):
    model.train()
    for x, y in tqdm(train_loader, desc=f"Ep {epoch+1}/{NUM_EPOCHS}", leave=False):
        x, y = x.cuda(), y.cuda()
        opt.zero_grad()
        crit(model(x), y).backward()
        opt.step()
    sched.step()

    model.eval()
    correct = 0; total = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.cuda(), y.cuda()
            pred = model(x).argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    acc = 100 * correct / total
    print(f"Epoch {epoch+1:2d}/{NUM_EPOCHS}: test acc = {acc:.2f}%")
    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), vgg_ckpt_path)

print(f"\nVGG11 training done. Best: {best_acc:.2f}%")

In [ ]:
# VGG11 sweep (this DID NOT finish - left for documentation)
# Ran for 24+ hours on T4 without producing a result
import os
os.environ["WANDB_MODE"] = "offline"
os.environ["WANDB_SILENT"] = "true"

%cd /content/ModelFolding

VGG_CKPT = f"{SAVE_DIR}/vgg11_bn_cifar10_torchvision.pt"

print("="*60); print("VGG11 Variant 1/3: Fold-AR (DF_REPAIR)"); print("="*60)
!python vgg11_cifar10_weight_clustering.py \
    --checkpoint {VGG_CKPT} \
    --repair "DF_REPAIR" \
    --proj_name "stress_test" --exp_name "vgg11_fold_ar"

print("\n" + "="*60); print("VGG11 Variant 2/3: Fold-Naive (NO_REPAIR)"); print("="*60)
!python vgg11_cifar10_weight_clustering.py \
    --checkpoint {VGG_CKPT} \
    --repair "NO_REPAIR" \
    --proj_name "stress_test" --exp_name "vgg11_fold_naive"

print("\n" + "="*60); print("VGG11 Variant 3/3: Fold-R (REPAIR)"); print("="*60)
!python vgg11_cifar10_weight_clustering.py \
    --checkpoint {VGG_CKPT} \
    --repair "REPAIR" \
    --proj_name "stress_test" --exp_name "vgg11_fold_r"

print("\n\nVGG11 sweeps done (if this line ever prints)")

---

End of notebook.

For the analysis and conclusions, see the final report in the `report/` folder of this repository.